# Results prediction model
In questo notebook viene mostrata la pipeline che abbiamo adottato per la creazione di un modello che possa predire i risultati delle partite.

## Caricamento dataset

Come al solito, recuperiamo il dataset completo.

In [18]:
import pandas as pd
import numpy as np

path = "static/da-result/result.csv"
dataset = pd.read_csv(path)

## Merge delle feature

Raggruppiamo le coppie di feature Home/Away in un unica feature data dal differenza tra feature home e feature away. Questo oltre a ridurre il numero di colonne, ci aiuta anche a ridurre la ridondanza e a semplificare il calcolo dei differenziali ai modelli. 

In [19]:
# Creo feature uniche
dataset["WinStreak"] = dataset["Home_WinStreak"] - dataset["Away_WinStreak"]
dataset["Z_Goals_Season"] = dataset["Z_Home_Goals_Season"] - dataset["Z_Away_Goals_Season"]
dataset["Z_Wins_Season"] = dataset["Z_Home_Wins_Season"] - dataset["Z_Away_Wins_Season"]
dataset["GoalOnShotRatio"] = dataset["GoalOnShotRatioHome"] - dataset["GoalOnShotRatioAway"]
dataset["PointToMatchRatio"] = dataset["PointToMatchRatioHome"] - dataset["PointToMatchRatioAway"]
dataset['Abs_Points_Difference'] = (dataset['Home_Current_Points'] - dataset['Away_Current_Points']).abs()
dataset["Elo"] = dataset["HomeElo"] - dataset["AwayElo"]
dataset["Abs_Value_Difference"] = (dataset["HomeValue"] - dataset["AwayValue"]).abs()
dataset["Value"] = dataset["HomeValue"] - dataset["AwayValue"]

# Variabile booleana che indica un match equilibrato sulla carta, ossia  
# se le due squadre hanno meno di 3 punti di distacco 
dataset['Is_Points_Balanced'] = (dataset['Abs_Points_Difference'] <= 3).astype(int)

# Elimino le feature inutilizzate
dataset = dataset.drop(columns=["Home_WinStreak", "Away_WinStreak", "Z_Home_Goals_Season", "Z_Away_Goals_Season", "Z_Home_Wins_Season", "Z_Away_Wins_Season", "GoalOnShotRatioHome", "GoalOnShotRatioAway", "PointToMatchRatioHome", "PointToMatchRatioAway", "HomeElo", "AwayElo", "HomeValue", "AwayValue"])

dataset = dataset.iloc[30:]

dataset.head()

,Date,HomeTeam,AwayTeam,FTHG,FTAG,FTR,Season,HomeAdvantage,Home_Current_Points,Away_Current_Points,WinStreak,Z_Goals_Season,Z_Wins_Season,GoalOnShotRatio,PointToMatchRatio,Abs_Points_Difference,Elo,Abs_Value_Difference,Value,Is_Points_Balanced
30,2015-09-19,Milan,Palermo,3,2,H,2015-2016,1.0,3,7,0,-0.642153,-0.795533,-0.333333,-1.0,4,-26.614687,71150000.0,71150000.0,0
31,2015-09-19,Udinese,Empoli,1,2,A,2015-2016,0.0,3,1,0,-0.967625,0.783747,-0.233333,0.5,2,17.363462,13050000.0,13050000.0,1
32,2015-09-20,Atalanta,Verona,1,1,D,2015-2016,1.0,4,2,0,0.428661,0.881792,-0.042569,0.5,2,22.163924,12250000.0,12250000.0,1
33,2015-09-20,Genoa,Juventus,0,2,A,2015-2016,1.0,3,1,0,0.005190,0.841378,-0.103175,0.5,2,17.450783,141500000.0,-141500000.0,1
34,2015-09-20,Bologna,Frosinone,1,0,H,2015-2016,0.0,0,0,0,0.033926,0.011803,-0.027778,0.0,0,12.154580,18000000.0,18000000.0,1


## Preparazione del target e split temporale

Procediamo:
- Mappando la colonna FTR in dati numerici (0 = Home, 1 = Draw, 2=Away)
- Ordinando cronologicamente il dataset
- Partizionandolo cronologicamente in set di training e set di test

In [20]:
from sklearn.metrics import accuracy_score, classification_report

# Mappo il target FTR in numeri
map = {'H': 0, 'D': 1, 'A': 2}
dataset['Numeric_target'] = dataset['FTR'].map(map)

# Ordino il dataset
dataset = dataset.sort_values(by='Date').reset_index(drop=True)

# Divido il traning set e test set in base alle stagioni (temporale)
test_mask = dataset['Season'] >= '2024-2025'
validation_mask = dataset['Season'] == '2023-2024'
train_mask = dataset['Season'] < '2023-2024'

# Separo i target usando le maschere
Y_train = dataset.loc[train_mask, 'Numeric_target']
Y_eval = dataset.loc[validation_mask, 'Numeric_target']
Y_test = dataset.loc[test_mask, 'Numeric_target']

# Separo le feature togliendo FTR
X = dataset.drop(columns=['FTR'])
X_train = X.loc[train_mask]
X_eval = X.loc[validation_mask]
X_test = X.loc[test_mask]


# Creiamo i 3 DataFrame completi
train_completo = X_train.copy()
train_completo['Numeric_target'] = Y_train

eval_completo = X_eval.copy()
eval_completo['Numeric_target'] = Y_eval

test_completo = X_test.copy()
test_completo['Numeric_target'] = Y_test

X_train.head()

,Date,HomeTeam,AwayTeam,FTHG,FTAG,Season,HomeAdvantage,Home_Current_Points,Away_Current_Points,WinStreak,Z_Goals_Season,Z_Wins_Season,GoalOnShotRatio,PointToMatchRatio,Abs_Points_Difference,Elo,Abs_Value_Difference,Value,Is_Points_Balanced,Numeric_target
0,2015-09-19,Milan,Palermo,3,2,2015-2016,1.0,3,7,0,-0.642153,-0.795533,-0.333333,-1.0,4,-26.614687,71150000.0,71150000.0,0,0
1,2015-09-19,Udinese,Empoli,1,2,2015-2016,0.0,3,1,0,-0.967625,0.783747,-0.233333,0.5,2,17.363462,13050000.0,13050000.0,1,2
2,2015-09-20,Atalanta,Verona,1,1,2015-2016,1.0,4,2,0,0.428661,0.881792,-0.042569,0.5,2,22.163924,12250000.0,12250000.0,1,1
3,2015-09-20,Genoa,Juventus,0,2,2015-2016,1.0,3,1,0,0.005190,0.841378,-0.103175,0.5,2,17.450783,141500000.0,-141500000.0,1,2
4,2015-09-20,Bologna,Frosinone,1,0,2015-2016,0.0,0,0,0,0.033926,0.011803,-0.027778,0.0,0,12.154580,18000000.0,18000000.0,1,0


### Random Baseline

Cominciamo osservando come un esperimento casuale si comporterebbe davanti alle partite del nostro test set (760). Questo passaggio lo utilizzamo per simulare lo scenario in cui un decisore (computer o un dado a tre facce), totalmente privo di dati storici o competenze calcistiche, tenti di indovinare l'esito dei match affidandosi al puro caso. 
L'output di questa simulazione fissa la nostra linea di partenza (random baseline) e diventa fondamentale per valutare il reale valore della ricerca.  

In [21]:
# Imposto il seed 
np.random.seed(42)

# Genero le predizioni totalmente casuali per le partite del test set
preds_casuali = np.random.choice([0, 1, 2], size=len(Y_test))

# Calcolo l'accuratezza del puro caso
accuratezza_puro_caso = accuracy_score(Y_test, preds_casuali)


print("\n==================================")
print("METRICHE DELL'ESPERIMENTO CASUALE")
print("==================================")
print(f"Accuratezza puro caso:    {accuratezza_puro_caso:.4f} ({accuratezza_puro_caso*100:.2f}%)")
print("\nReport dettagliato del Modello Casuale Puro:")
print(classification_report(Y_test, preds_casuali, target_names=['1', 'X', '2'], zero_division=0))


METRICHE DELL'ESPERIMENTO CASUALE
Accuratezza puro caso:    0.3283 (32.83%)

Report dettagliato del Modello Casuale Puro:
              precision    recall  f1-score   support

           1       0.39      0.36      0.37       291
           X       0.26      0.31      0.28       202
           2       0.32      0.31      0.32       241

    accuracy                           0.33       734
   macro avg       0.33      0.33      0.32       734
weighted avg       0.33      0.33      0.33       734



## Scelta delle feature
In questa sezione viene creata una classe che, dati in input le feature di interesse, utilizza l'interpolazione logistica per calcolare le metriche del modello, in funzione delle feautre che utilizziamo ingresso.

In [22]:
from pandas import DataFrame
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, log_loss, classification_report

class LinearRegressionTester:

    __train_set: DataFrame
    __val_set: DataFrame
    __test_set: DataFrame

    def __init__(self, train_set: DataFrame, val_set: DataFrame, test_set: DataFrame):
        """
        :param train_set: Set di training, usato per addestrare il modello
        :param val_set: Set di validazione, usato per la selezione degli iperparametri
        :param test_set: Set di test, usato per la valutazione finale
        """
        self.__train_set = train_set
        self.__val_set = val_set
        self.__test_set = test_set

    def run_logistic_regression_baseline(
        self,
        target_col: str,
        baseline_features: list[str]
    ):
        """
        Esegue la baseline lineare (Regressione Logistica) con selezione
        dell'iperparametro C tramite il validation set, poi valuta sul test set.

        :param target_col: Il nome della colonna contenente le etichette target (Y)
        :param baseline_features: lista di feature con le quali viene valutato il modello
        """

        C_values = [0.001, 0.01, 0.1, 1.0, 10.0, 100.0]

        # Estrazione delle feature di interesse 
        X_train_features = self.__train_set[baseline_features]
        Y_train = self.__train_set[target_col]

        X_val_features = self.__val_set[baseline_features]
        Y_val = self.__val_set[target_col]

        X_test_features = self.__test_set[baseline_features]
        Y_test = self.__test_set[target_col]

        # Standardizzazione 
        scaler = StandardScaler()
        X_train_scaled = scaler.fit_transform(X_train_features)
        X_val_scaled = scaler.transform(X_val_features)
        X_test_scaled = scaler.transform(X_test_features)

        # Scetla degli iperparametri per evitare overfitting
        best_C = None
        best_val_score = -1.0
        best_model = None

        for C in C_values:
            model = LogisticRegression(
                solver='lbfgs',
                max_iter=500,
                random_state=42,
                C=C
            )
            model.fit(X_train_scaled, Y_train)

            val_preds = model.predict(X_val_scaled)
            val_acc = accuracy_score(Y_val, val_preds)

            if val_acc > best_val_score:
                best_val_score = val_acc
                best_C = C
                best_model = model

        base_preds = best_model.predict(X_test_scaled)
        base_probs = best_model.predict_proba(X_test_scaled)

        #print(f"\n=== Risultati sul Test Set (C={best_C}) ===")
        #print(f"Accuratezza (Precision): {accuracy_score(Y_test, base_preds):.4f} ({accuracy_score(Y_test, base_preds)*100:.2f}%)")
        #print(f"Log-Loss:    {log_loss(Y_test, base_probs):.4f}")
        #print("\nReport:")
        #print(classification_report(Y_test, base_preds, target_names=['1', 'X', '2'], zero_division=0))

        return best_model, scaler, best_C


    def ottimizza_pareggio_validation(self, model, scaler, target_col: str, baseline_features: list[str]):
        """
            Trova la soglia ottimale per il pareggio analizzando il Validation Set.
            Applica una ricerca a griglia per fare in modo che il modello generalizzi al meglio,
            e una volta trovata la miglior soglia, la applica al Test Set.

        """
        from sklearn.metrics import f1_score, accuracy_score, classification_report
        
        # Estrazione e scaling del validation Set
        X_val_scaled = scaler.transform(self.__val_set[baseline_features])
        Y_val = self.__val_set[target_col].values

        # Estrazione e scaling del test Set 
        X_test_scaled = scaler.transform(self.__test_set[baseline_features])
        Y_test = self.__test_set[target_col].values

        # Ottengo le probabilità calcolate dal modello
        val_probs = model.predict_proba(X_val_scaled)
        test_probs = model.predict_proba(X_test_scaled)

        # Griglia di soglie da testare per il pareggio (da 0.20 a 0.40 con passi di 0.01)
        soglie_da_testare = np.arange(0.20, 0.41, 0.01)
        
        best_threshold = 0.33
        best_val_f1 = -1.0
        best_val_acc = -1.0

        print("--- Fase di Ottimizzazione sul Validation Set ---")
        
        for thresh in soglie_da_testare:
            val_preds = []
            for p in val_probs:
                # Se la probabilità del pareggio (p[1]) supera la soglia, predici X 
                if p[1] >= thresh:
                    val_preds.append(1)
                else:
                    # Altrimenti prendiamo la probabilità più alta tra Casa (0) e Trasferta (2)
                    val_preds.append(0 if p[0] > p[2] else 2)
            
            # Calcoliamo il macro F1-score (ottimo per bilanciare l'accuratezza su tutte e tre le classi)
            current_f1 = f1_score(Y_val, val_preds, average='macro', zero_division=0)
            current_acc = accuracy_score(Y_val, val_preds)

            # Scegliamo la soglia ottima
            if current_f1 > best_val_f1:
                best_val_f1 = current_f1
                best_val_acc = current_acc
                best_threshold = thresh

        print(f"\n[SOGLIA OTTIMIZZATA]: {best_threshold:.2f}")
        print(f"Accuratezza sul Validation Set con questa soglia: {best_val_acc:.2%}")
        print(f"Macro F1-Score sul Validation Set: {best_val_f1:.4f}")
        print("=========================================================")

        # Applichiamo la soglia al test set per vedere se generalizza in modo corretto
        final_test_preds = []
        for p in test_probs:
            if p[1] >= best_threshold:
                final_test_preds.append(1)
            else:
                final_test_preds.append(0 if p[0] > p[2] else 2)

        print(f"\n=== VERIFICA DI GENERALIZZAZIONE SUL TEST SET (Soglia X = {best_threshold:.2f}) ===")
        print(f"Accuratezza Finale sul Test Set: {accuracy_score(Y_test, final_test_preds):.2%}")
        print(f"Log-Loss:    {log_loss(Y_test, test_probs):.4f}")
        

        print("\nReport di Classificazione sul Test Set:")
        print(classification_report(Y_test, final_test_preds, target_names=['1', 'X', '2'], zero_division=0))
        
        # Ritorniamo le predizioni finali 
        return final_test_preds
    


## Scelta delle feature con regressione logistica
Adesso, tramite la classe appena creata, scegliamo delle feature e valutiamo l'impatto sulle metriche, eliminandone e aggiungendone di nuove, seguendo anche la heatmap del notebook feature-choice. Per prima cosa creaiamo un'istanza dalla classe, che successivamente andremo ad utilizzare.

In [23]:
ml = LinearRegressionTester(train_completo, eval_completo, test_completo)

### Baseline con regressione logistica 

Realizziamo adesso una baseline basata sul modello della regressione logistica. Per farlo utilizziamo delle feature base da noi calcolate, come la striscia di vittorie, il rapporto punti/match ed altri. 

In [24]:
baseline = ["WinStreak","GoalOnShotRatio","Elo"] # Feature Base

best_model, scaler, best_C = ml.run_logistic_regression_baseline("Numeric_target", baseline)

predizioni_personalizzate = ml.ottimizza_pareggio_validation(
    model=best_model,
    scaler=scaler,
    target_col='Numeric_target',
    baseline_features= baseline
)


--- Fase di Ottimizzazione sul Validation Set ---

[SOGLIA OTTIMIZZATA]: 0.27
Accuratezza sul Validation Set con questa soglia: 53.95%
Macro F1-Score sul Validation Set: 0.4943

=== VERIFICA DI GENERALIZZAZIONE SUL TEST SET (Soglia X = 0.27) ===
Accuratezza Finale sul Test Set: 52.72%
Log-Loss:    0.9884

Report di Classificazione sul Test Set:
              precision    recall  f1-score   support

           1       0.53      0.75      0.62       291
           X       0.35      0.16      0.22       202
           2       0.59      0.56      0.58       241

    accuracy                           0.53       734
   macro avg       0.49      0.49      0.47       734
weighted avg       0.50      0.53      0.50       734



si ha un ovvio miglioramento rispetto alla random baseline, sia per quanto riguarda l'accuratezza che per la precision e la recall. Sebbene la recall dei pareggi diminuisca

#### L'influenza dell'home advantage

Per capire se l'introduzione del **Fattore Campo (Home Advantage)** migliori effettivamente le performance del nostro modello di predizione, osserivamo le metriche una volta aggiunta questa feature

In [25]:
features_scelte = baseline + ["HomeAdvantage"]

best_model, scaler, best_C = ml.run_logistic_regression_baseline("Numeric_target", features_scelte)

predizioni_personalizzate = ml.ottimizza_pareggio_validation(
    model=best_model,
    scaler=scaler,
    target_col='Numeric_target',
    baseline_features= features_scelte
)


--- Fase di Ottimizzazione sul Validation Set ---

[SOGLIA OTTIMIZZATA]: 0.27
Accuratezza sul Validation Set con questa soglia: 50.79%
Macro F1-Score sul Validation Set: 0.4900

=== VERIFICA DI GENERALIZZAZIONE SUL TEST SET (Soglia X = 0.27) ===
Accuratezza Finale sul Test Set: 50.00%
Log-Loss:    0.9871

Report di Classificazione sul Test Set:
              precision    recall  f1-score   support

           1       0.55      0.62      0.58       291
           X       0.32      0.34      0.33       202
           2       0.61      0.50      0.55       241

    accuracy                           0.50       734
   macro avg       0.49      0.48      0.49       734
weighted avg       0.51      0.50      0.50       734



Confrontando gli ultimi due report delle metriche ottenuti, notiamo che:

* **Vittoria in Casa:** La precision è salita nettamente dal **44% al 51%**. Il modello è diventato molto più affidabile quando pronostica la vittoria della squadra ospitante.
* **Pareggio:** Rimane l'esito più difficile da prevedere, ma si nota un leggero incremento della precision (da 22% a 36%) e dell'F1-score.
* **Vittoria in Trasferta:** Il recall è salito dal **46% al 59%**, segno che il modello riesce a catturare molte più vittorie esterne, probabilmente identificando meglio quando una squadra forte riesce a superare lo svantaggio del campo avversario.

> **Conclusione:** La feature *Home Advantage* ha un ruolo **significativo e positivo** nel modello. Non solo ha migliorato l'accuratezza generale del 4%, ma ha ridotto pure la log-loss, confermandosi una variabile importante per la predizione degli eventi sportivi.

#### PointToMatchRatio 
Facciamo un ulteriore passo avanti nel raffinamento del dataset introducendo una variabile essenziale per il rendimento delle squadre: il **Points Per Match (PPM) Ratio** (o *Media Punti per Partita*).

In [26]:
features_scelte = baseline + ["HomeAdvantage","PointToMatchRatio"]

best_model, scaler, best_C = ml.run_logistic_regression_baseline("Numeric_target", features_scelte)

predizioni_personalizzate = ml.ottimizza_pareggio_validation(
    model=best_model,
    scaler=scaler,
    target_col='Numeric_target',
    baseline_features= features_scelte
)


--- Fase di Ottimizzazione sul Validation Set ---

[SOGLIA OTTIMIZZATA]: 0.27
Accuratezza sul Validation Set con questa soglia: 51.05%
Macro F1-Score sul Validation Set: 0.4929

=== VERIFICA DI GENERALIZZAZIONE SUL TEST SET (Soglia X = 0.27) ===
Accuratezza Finale sul Test Set: 49.73%
Log-Loss:    0.9873

Report di Classificazione sul Test Set:
              precision    recall  f1-score   support

           1       0.55      0.61      0.58       291
           X       0.31      0.34      0.33       202
           2       0.61      0.49      0.55       241

    accuracy                           0.50       734
   macro avg       0.49      0.48      0.48       734
weighted avg       0.51      0.50      0.50       734



L'integrazione del PPM Ratio ha influito positivamente su tutti e tre gli esiti:

* **Vittoria in Casa:** La precision sale ancora arrivando al **53%**, accoppiata a un ottimo recall del **55%**. Il modello ora intercetta benissimo le favorite in casa.
* **Pareggio:** Registra un incremento importante: la precision sale al **34%** (+4%) e l'F1-score passa da 0.24 a **0.28**.
* **Vittoria in Trasferta:** È l'aspetto che è migliorato maggiormente. La precision passa ad essere del **52%** (era al 41%) e il recall tocca il **61%**. Il PPM Ratio permette quindi al modello di capire quando una squadra ospite è statisticamente dominante, indipendentemente dal fattore campo.

> **Conclusione:** Il *Point to Match Ratio* si è rivelato la feature più importante inserita finora, registrando un miglioramento del 7% sull'accuratezza e una drastica riduzione della log-loss

#### Aggiunta dello z-score

Per fare un ulteriore salto di qualità rispetto alla baseline, introduciamo nel dataset le feature basate sullo **Z-Score**. Questa metrica esprime lo scostamento di una specifica caratteristica dalla media del campionato, misurato in unità di deviazioni standard.
L'introduzione dello Z-Score è fondamentale per catturare la **dinamica temporale** del campionato, risolvendo un limite intrinseco dei dati grezzi. Due valori numericamente identici possono infatti assumere significati opposti a seconda del momento in cui si verificano.


In [27]:
features_scelte = baseline + ["HomeAdvantage","Z_Goals_Season"]

best_model, scaler, best_C = ml.run_logistic_regression_baseline("Numeric_target", features_scelte)

predizioni_personalizzate = ml.ottimizza_pareggio_validation(
    model=best_model,
    scaler=scaler,
    target_col='Numeric_target',
    baseline_features= features_scelte
)


--- Fase di Ottimizzazione sul Validation Set ---

[SOGLIA OTTIMIZZATA]: 0.27
Accuratezza sul Validation Set con questa soglia: 52.37%
Macro F1-Score sul Validation Set: 0.4984

=== VERIFICA DI GENERALIZZAZIONE SUL TEST SET (Soglia X = 0.27) ===
Accuratezza Finale sul Test Set: 50.00%
Log-Loss:    0.9883

Report di Classificazione sul Test Set:
              precision    recall  f1-score   support

           1       0.54      0.66      0.59       291
           X       0.31      0.25      0.28       202
           2       0.59      0.51      0.55       241

    accuracy                           0.50       734
   macro avg       0.48      0.48      0.47       734
weighted avg       0.49      0.50      0.49       734



### Aggiunta della differenza punti 
Integriamo adesso l'ultima feature, ovvero la differenza punti in valore assoluto.

In [28]:
features_scelte = baseline + ["HomeAdvantage","Z_Goals_Season","Abs_Points_Difference"]

best_model, scaler, best_C = ml.run_logistic_regression_baseline("Numeric_target", features_scelte)

predizioni_personalizzate = ml.ottimizza_pareggio_validation(
    model=best_model,
    scaler=scaler,
    target_col='Numeric_target',
    baseline_features= features_scelte
)


--- Fase di Ottimizzazione sul Validation Set ---

[SOGLIA OTTIMIZZATA]: 0.27
Accuratezza sul Validation Set con questa soglia: 52.11%
Macro F1-Score sul Validation Set: 0.5097

=== VERIFICA DI GENERALIZZAZIONE SUL TEST SET (Soglia X = 0.27) ===
Accuratezza Finale sul Test Set: 51.50%
Log-Loss:    0.9874

Report di Classificazione sul Test Set:
              precision    recall  f1-score   support

           1       0.57      0.60      0.58       291
           X       0.36      0.40      0.37       202
           2       0.61      0.51      0.56       241

    accuracy                           0.51       734
   macro avg       0.51      0.50      0.51       734
weighted avg       0.52      0.51      0.52       734



### Modelli non lineari
In questa sezione utilizzeremo due modelli non lineari, ovvero la **discesa del gradiente** e la **random forest** per allenare il modello, con le intuizioni che abbiamo sviluppato osservando i risultati della regressione logistica. Per la discesa del gradiente utilizziamo la libreria **SGDClassifier** (lineare) e **GradientBoostingClassifier** di scikit-learn.

In [29]:
from sklearn.pipeline import make_pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score,f1_score
from xgboost import XGBClassifier

class NonLinearModels:

    __X_train_features: DataFrame
    __Y_train_result: DataFrame
    __X_val_features: DataFrame
    __Y_val_result: DataFrame
    __X_test_features: DataFrame
    __Y_test_result: DataFrame

    def __init__(self, train_set: DataFrame, val_set: DataFrame, test_set: DataFrame,
                 features: list[str], target: str):

        # Train
        self.__X_train_features = train_set[features]
        self.__Y_train_result = train_set[target]

        # Validation
        self.__X_val_features = val_set[features]
        self.__Y_val_result = val_set[target]

        # Test
        self.__X_test_features = test_set[features]
        self.__Y_test_result = test_set[target]

    
    def __print_model_evaluation(self, predictions, probabilities):
        """
        Metodo che stampa i risultati del modello
        """

        # Metriche di Classificazione
        accuracy = accuracy_score(self.__Y_test_result, predictions)
        report = classification_report(self.__Y_test_result, predictions)
        loss_value = log_loss(self.__Y_test_result, probabilities)

        print("--- Modello di Classificazione Addestrato ---")
        print(f"Accuracy Globale: {accuracy * 100:.2f}%\n")
        print(f"Log Loss: {loss_value:.4f}")
        print("Dettaglio per ogni classe (0=Casa, 1=Pareggio, 2=Trasferta):")
        print(report)

    
    def random_forest(self):
        """
        Metodo che addestra un modello RandomForestClassifier, valutando anche gli iperparametri
        """

        param_grid = {
            "n_estimators": [100, 200, 300],
            "max_depth": [None, 5, 10],
            "min_samples_leaf": [1, 2, 4]
        }

        best_params = None
        best_val_score = -1.0
        best_model = None

        for n_estimators in param_grid["n_estimators"]:
            for max_depth in param_grid["max_depth"]:
                for min_samples_leaf in param_grid["min_samples_leaf"]:

                    model = RandomForestClassifier(
                        n_estimators=n_estimators,
                        max_depth=max_depth,
                        min_samples_leaf=min_samples_leaf,
                        random_state=42,
                        n_jobs=-1
                    )
                    
                    model.fit(self.__X_train_features, self.__Y_train_result)

                    val_preds = model.predict(self.__X_val_features)
                    val_acc = accuracy_score(self.__Y_val_result, val_preds)

                    if val_acc > best_val_score:
                        best_val_score = val_acc
                        best_params = {
                            "n_estimators": n_estimators,
                            "max_depth": max_depth,
                            "min_samples_leaf": min_samples_leaf
                        }
                        best_model = model


        return best_model, best_params
    

    def xgboost_classifier(self):
        """
            Metodo che addestra un modello XGBClassifier, valutando anche gli iperparametri
            sul validation set, per poi testarlo sul test set.
        """

        # Iperparametri da testare
        param_grid = {
            "n_estimators": [100, 150],
            "max_depth": [3, 4],            
            "learning_rate": [0.03, 0.05],
            "subsample": [0.8, 1.0],         
            "colsample_bytree": [0.8, 1.0],  
            "reg_alpha": [0.1, 1.0],         
            "reg_lambda": [1.0, 5.0]         
        }

        best_params = None
        best_val_score = -1.0
        best_model = None

        # Ciclo per la ricerca dei migliori iperparametri
        for n_estimators in param_grid["n_estimators"]:
            for max_depth in param_grid["max_depth"]:
                for learning_rate in param_grid["learning_rate"]:


                    model = XGBClassifier(
                        n_estimators=n_estimators,
                        max_depth=max_depth,
                        learning_rate=learning_rate,
                        random_state=42,
                        eval_metric='mlogloss',
                        n_jobs=-1
                    )

                    model.fit(self.__X_train_features, self.__Y_train_result)

                    # Ci basiamo sull'F1 score
                    val_preds = model.predict(self.__X_val_features)
                    val_f1_macro = f1_score(self.__Y_val_result, val_preds, average='macro')

                    # Seleziono il modello basandomi sul miglior F1-Score Macro
                    if val_f1_macro > best_val_score:
                        best_val_score = val_f1_macro
                        best_params = { ... }
                        best_model = model

        return best_model, best_params


    def ottimizza_soglia_pareggio(self, model):
        """
            Trova la soglia ottimale per il pareggio (X) direttamente sul Validation Set,
            massimizzando la metrica Macro F1-Score.
            Valuta infine il risultato sul Test Set finale.
        """
        from sklearn.metrics import f1_score, accuracy_score, classification_report, log_loss
        import numpy as np

        # Probabilità sul validation set
        val_probs = model.predict_proba(self.__X_val_features)
        Y_val = self.__Y_val_result.values

        # Griglia di soglie da testare per il segno X 
        soglie_da_testare = np.arange(0.20, 0.41, 0.01)
        
        best_threshold = 0.33
        best_score = -1.0
        
        print("--- Ottimizzazione Soglia su Validation Set Standard ---")
    
        # Ciclo sulle soglie per trovare la migliore sul Validation Set
        for thresh in soglie_da_testare:
            val_preds = []
            for p in val_probs:
                if p[1] >= thresh:
                    val_preds.append(1)  # Forza Pareggio (X)
                else:
                    val_preds.append(0 if p[0] > p[2] else 2) # Segno più probabile tra 1 e 2
            
            f1 = f1_score(Y_val, val_preds, average='macro', zero_division=0)
            punteggio_combinato = f1 # Massimizza esclusivamente l'F1-Score Macro
            
            if punteggio_combinato > best_score:
                best_score = punteggio_combinato
                best_threshold = thresh

        print("=========================================================")
        print(f"[SOGLIA VAL]: {best_threshold:.2f}")
        print(f"Miglior Macro F1-Score su Validation: {best_score:.4f}")
        print("=========================================================\n")

        # Valutazione sul test set
        test_probs = model.predict_proba(self.__X_test_features)
        Y_test = self.__Y_test_result.values
        
        final_test_preds = []
        for p in test_probs:
            if p[1] >= best_threshold:
                final_test_preds.append(1)
            else:
                final_test_preds.append(0 if p[0] > p[2] else 2)

        # Metriche sul Test Set
        accuracy = accuracy_score(Y_test, final_test_preds)
        loss_value = log_loss(Y_test, test_probs)
        report = classification_report(Y_test, final_test_preds, target_names=['1', 'X', '2'], zero_division=0)

        print(f"=== VERIFICA DI GENERALIZZAZIONE SUL TEST SET FINALE (Soglia = {best_threshold:.2f}) ===")
        print(f"Accuracy Globale: {accuracy * 100:.2f}%")
        print(f"Log Loss:         {loss_value:.4f}")
        print("\nReport di Classificazione:")
        print(report)
        
        return final_test_preds

In [30]:
nlm = NonLinearModels(X_train, X_eval, X_test, ["Value", "Z_Wins_Season", "Abs_Value_Difference"], "Numeric_target")

In [31]:
# 1. Allena il modello (es. RandomForest o XGBoost)
best_xg, best_xg_params = nlm.xgboost_classifier()

# 2. Trova la soglia sul Validation standard e verifica sul Test set
predizioni_finali = nlm.ottimizza_soglia_pareggio(model=best_xg)

--- Ottimizzazione Soglia su Validation Set Standard ---
[SOGLIA VAL]: 0.29
Miglior Macro F1-Score su Validation: 0.5082

=== VERIFICA DI GENERALIZZAZIONE SUL TEST SET FINALE (Soglia = 0.29) ===
Accuracy Globale: 52.04%
Log Loss:         0.9934

Report di Classificazione:
              precision    recall  f1-score   support

           1       0.57      0.61      0.59       291
           X       0.37      0.32      0.34       202
           2       0.57      0.58      0.58       241

    accuracy                           0.52       734
   macro avg       0.50      0.50      0.50       734
weighted avg       0.51      0.52      0.52       734



In [32]:
best_rf, best_rf_params = nlm.random_forest()

# 2. Trova la soglia sul Validation standard e verifica sul Test set
predizioni_finali = nlm.ottimizza_soglia_pareggio(model=best_rf)

--- Ottimizzazione Soglia su Validation Set Standard ---
[SOGLIA VAL]: 0.29
Miglior Macro F1-Score su Validation: 0.5186

=== VERIFICA DI GENERALIZZAZIONE SUL TEST SET FINALE (Soglia = 0.29) ===
Accuracy Globale: 53.81%
Log Loss:         0.9856

Report di Classificazione:
              precision    recall  f1-score   support

           1       0.58      0.62      0.60       291
           X       0.40      0.36      0.38       202
           2       0.59      0.60      0.59       241

    accuracy                           0.54       734
   macro avg       0.52      0.52      0.52       734
weighted avg       0.53      0.54      0.53       734



## Salvataggio dei modelli
In questa parte finale del documento verranno esportati i modelli in formato .joblib, per essere riutilizzati nell'applicazione.

In [33]:
def save_scikit_model(
    model_var,
    model_path,
    scaler = None,
    features = None    
):
    """
    Funzione che permette di salvare ed esportare i modelli scikit in formato joblib
    """
    import joblib
    
    if scaler:
        joblib.dump({'model': model_var, 'scaler': scaler, 'features': features}, model_path)
    else:
        joblib.dump({'model': model_var}, model_path)



In [34]:
# Esportazione della regressione logistica
save_scikit_model(best_model, "static/models/linear_model.joblib", scaler, baseline + ["HomeAdvantage","Z_Goals_Season","Abs_Points_Difference"])

# Random Forest
save_scikit_model(best_rf, "static/models/random_forest_model.joblib")

# XgdBoost
save_scikit_model(best_xg, "static/models/random_xgboost_model.joblib")